# 01 - Segmentación de Clientes

## Pregunta de negocio: ¿Qué segmentos naturales de clientes existen?

---

En este notebook aplicamos **aprendizaje no supervisado** para descubrir grupos naturales de clientes
a partir de datos de comportamiento de conducción y características demográficas.

### Fundamentos teóricos

#### K-Means
K-Means es un algoritmo de clustering particional que divide los datos en **k grupos** minimizando la
**inercia** (suma de distancias al cuadrado de cada punto a su centroide más cercano).

Funcionamiento:
1. Se eligen k centroides iniciales de forma aleatoria (o con `k-means++`).
2. Cada observación se asigna al centroide más cercano.
3. Se recalculan los centroides como la media de las observaciones asignadas.
4. Se repiten los pasos 2-3 hasta convergencia.

**Inercia**: $W = \sum_{j=1}^{k} \sum_{x_i \in C_j} ||x_i - \mu_j||^2$

Limitaciones: requiere definir k de antemano, asume clusters esféricos, sensible a outliers.

#### DBSCAN (Density-Based Spatial Clustering of Applications with Noise)
DBSCAN agrupa puntos que están densamente empaquetados. No requiere especificar el número de clusters.

Parámetros clave:
- **eps (ε)**: radio máximo de vecindad.
- **min_samples**: número mínimo de puntos para formar una región densa.

Ventajas: detecta clusters de forma arbitraria, identifica **puntos de ruido** (outliers), no necesita k.

#### Coeficiente de Silueta (Silhouette Score)
Mide qué tan similar es un punto a su propio cluster comparado con el cluster más cercano.

$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$

Donde $a(i)$ es la distancia media intra-cluster y $b(i)$ la distancia media al cluster más cercano.
Rango: [-1, 1]. Valores cercanos a 1 indican buena separación.

#### PCA (Análisis de Componentes Principales)
Técnica de **reducción de dimensionalidad** que transforma las variables originales en componentes
ortogonales que capturan la máxima varianza. Nos permite visualizar datos multidimensionales en 2D/3D
y entender qué variables contribuyen más a la estructura de los datos.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import FancyBboxPatch
from math import pi
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

vtype_colors = {
    'electrico': '#2ecc71',
    'gasolina': '#3498db',
    'hibrido': '#9b59b6',
    'deportivo': '#e74c3c'
}

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
print(f"Directorio raíz del proyecto: {project_root}")

---
## 2. Carga y preparación de datos

Cargamos el dataset combinado de features de conducción y datos demográficos.
Seleccionamos variables relevantes para la segmentación y aplicamos **StandardScaler**.

### ¿Por qué normalizar?
Los algoritmos de clustering basados en distancia (K-Means, DBSCAN) son sensibles a la escala.
Si una variable tiene rango [0, 100000] y otra [0, 1], la primera dominará las distancias.
StandardScaler transforma cada variable a media 0 y desviación estándar 1, asegurando que
todas contribuyan equitativamente.

In [ ]:
# Intentar cargar vehicle_survey_merged.csv, si no existe usar features_ml.csv
merged_path = os.path.join(project_root, "data/processed/vehicle_survey_merged.csv")
features_path = os.path.join(project_root, "data/processed/features_ml.csv")

if os.path.exists(merged_path):
    df = pd.read_csv(merged_path)
    print(f"Cargado vehicle_survey_merged.csv: {df.shape}")
else:
    df = pd.read_csv(features_path)
    print(f"Cargado features_ml.csv: {df.shape}")

print(f"\nColumnas disponibles ({len(df.columns)}):")
print(df.columns.tolist())
df.head()

In [ ]:
# Seleccionar features para clustering (comportamiento + demográficas)
# Adaptamos según las columnas disponibles
behavior_candidates = [
    'speed_mean', 'speed_std', 'acceleration_std', 'consumption_mean',
    'harsh_braking_count', 'harsh_accel_count', 'night_ratio', 'highway_ratio',
    'avg_speed_kmh', 'std_speed_kmh', 'avg_consumption', 'max_speed_kmh'
]
demographic_candidates = [
    'age', 'satisfaction_score', 'income_numeric',
    'income_bracket_encoded', 'would_recommend'
]

available_behavior = [c for c in behavior_candidates if c in df.columns]
available_demographic = [c for c in demographic_candidates if c in df.columns]

# Si no hay suficientes features específicas, usar todas las numéricas
if len(available_behavior) + len(available_demographic) < 4:
    cluster_features = df.select_dtypes(include=[np.number]).columns.tolist()
    # Excluir IDs y columnas no informativas
    exclude_cols = ['vehicle_id', 'trip_id', 'gps_lat', 'gps_lon']
    cluster_features = [c for c in cluster_features if c not in exclude_cols]
else:
    cluster_features = available_behavior + available_demographic

print(f"Features seleccionadas para clustering ({len(cluster_features)}):")
print(cluster_features)

# Preparar matriz de clustering
df_cluster = df[cluster_features].dropna()
print(f"\nObservaciones para clustering: {len(df_cluster)}")

# Normalizar con StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)

print(f"\nVerificación de normalización:")
print(f"  Media por feature (deben ser ~0): {X_scaled.mean(axis=0).round(4)[:5]}...")
print(f"  Std por feature (deben ser ~1):   {X_scaled.std(axis=0).round(4)[:5]}...")

---
## 3. Determinación del número óptimo de clusters

Usamos dos métodos complementarios:

- **Método del codo (Elbow)**: buscamos el punto donde la inercia deja de decrecer significativamente.
- **Coeficiente de silueta**: buscamos el k que maximice la separación entre clusters.

In [ ]:
k_range = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))
    print(f"  k={k}: Inercia={km.inertia_:.1f}, Silueta={silhouette_scores[-1]:.4f}")

# Visualizar ambos métodos lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Número de clusters (k)', fontsize=12)
axes[0].set_ylabel('Inercia', fontsize=12)
axes[0].set_title('Método del Codo', fontsize=14, fontweight='bold')
axes[0].set_xticks(list(k_range))

# Silhouette
best_k = list(k_range)[np.argmax(silhouette_scores)]
colors_sil = ['#e74c3c' if k == best_k else '#3498db' for k in k_range]
axes[1].bar(k_range, silhouette_scores, color=colors_sil, edgecolor='white')
axes[1].set_xlabel('Número de clusters (k)', fontsize=12)
axes[1].set_ylabel('Coeficiente de Silueta', fontsize=12)
axes[1].set_title('Coeficiente de Silueta por k', fontsize=14, fontweight='bold')
axes[1].set_xticks(list(k_range))

fig.suptitle('Determinación del número óptimo de clusters', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nMejor k según silueta: {best_k} (score={max(silhouette_scores):.4f})")
optimal_k = best_k

---
## 4. K-Means Clustering

Ajustamos K-Means con el k óptimo identificado y visualizamos los clusters
proyectados en 2D mediante PCA.

In [ ]:
# Ajustar K-Means con k óptimo
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_cluster = df_cluster.copy()
df_cluster['cluster_kmeans'] = kmeans.fit_predict(X_scaled)

print(f"K-Means con k={optimal_k}")
print(f"Inercia final: {kmeans.inertia_:.2f}")
print(f"Silueta: {silhouette_score(X_scaled, df_cluster['cluster_kmeans']):.4f}")
print(f"\nDistribución de clusters:")
print(df_cluster['cluster_kmeans'].value_counts().sort_index())

In [ ]:
# Visualización PCA 2D de clusters K-Means
pca_2d = PCA(n_components=2, random_state=42)
X_pca = pca_2d.fit_transform(X_scaled)

# Proyectar centroides
centers_pca = pca_2d.transform(kmeans.cluster_centers_)

cluster_colors = plt.cm.Set2(np.linspace(0, 1, optimal_k))

fig, ax = plt.subplots(figsize=(10, 8))
for i in range(optimal_k):
    mask = df_cluster['cluster_kmeans'] == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=[cluster_colors[i]], label=f'Cluster {i}',
               alpha=0.6, s=50, edgecolors='white', linewidth=0.5)

# Centroides
ax.scatter(centers_pca[:, 0], centers_pca[:, 1],
           c='black', marker='X', s=200, linewidths=2,
           edgecolors='white', zorder=5, label='Centroides')

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% varianza)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% varianza)', fontsize=12)
ax.set_title('Clusters K-Means (proyección PCA 2D)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='best')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de centros de clusters
centers_df = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=cluster_features,
    index=[f'Cluster {i}' for i in range(optimal_k)]
)

fig, ax = plt.subplots(figsize=(14, max(4, optimal_k * 1.2)))
sns.heatmap(centers_df, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, center=centers_df.values.mean())
ax.set_title('Centros de Clusters K-Means (valores originales)', fontsize=14, fontweight='bold')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()

---
## 5. DBSCAN

DBSCAN no requiere especificar el número de clusters. Usamos el método de **k-distancias**
(NearestNeighbors) para estimar un buen valor de **eps**: graficamos la distancia al k-ésimo
vecino más cercano ordenada de forma ascendente y buscamos el "codo" de la curva.

In [ ]:
# Estimación de eps con NearestNeighbors
min_samples = max(2 * len(cluster_features), 5)  # regla práctica: 2 * dimensiones

nn = NearestNeighbors(n_neighbors=min_samples)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)

# Distancias al k-ésimo vecino, ordenadas
k_distances = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(len(k_distances)), k_distances, 'b-', linewidth=1.5)
ax.set_xlabel('Índice (ordenado)', fontsize=12)
ax.set_ylabel(f'Distancia al {min_samples}-ésimo vecino', fontsize=12)
ax.set_title('Gráfico de k-distancias para estimación de eps', fontsize=14, fontweight='bold')

# Estimar eps automáticamente: buscar el punto de mayor curvatura
# Aproximación: usar el percentil 90 de las distancias
eps_estimated = np.percentile(k_distances, 90)
ax.axhline(y=eps_estimated, color='r', linestyle='--', label=f'eps estimado = {eps_estimated:.2f}')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"Parámetros DBSCAN: eps={eps_estimated:.3f}, min_samples={min_samples}")

In [ ]:
# Ajustar DBSCAN
dbscan = DBSCAN(eps=eps_estimated, min_samples=min_samples)
df_cluster['cluster_dbscan'] = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(df_cluster['cluster_dbscan'])) - (1 if -1 in df_cluster['cluster_dbscan'].values else 0)
n_noise = (df_cluster['cluster_dbscan'] == -1).sum()

print(f"DBSCAN encontró {n_clusters_db} clusters")
print(f"Puntos de ruido (outliers): {n_noise} ({n_noise/len(df_cluster)*100:.1f}%)")
print(f"\nDistribución de clusters DBSCAN:")
print(df_cluster['cluster_dbscan'].value_counts().sort_index())

# Silueta (excluyendo ruido)
mask_no_noise = df_cluster['cluster_dbscan'] != -1
if mask_no_noise.sum() > 0 and n_clusters_db > 1:
    sil_db = silhouette_score(X_scaled[mask_no_noise], df_cluster.loc[mask_no_noise, 'cluster_dbscan'])
    print(f"Silueta DBSCAN (sin ruido): {sil_db:.4f}")

In [ ]:
# Comparación visual K-Means vs DBSCAN
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# K-Means
for i in range(optimal_k):
    mask = df_cluster['cluster_kmeans'] == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[cluster_colors[i]], label=f'Cluster {i}',
                    alpha=0.6, s=40, edgecolors='white', linewidth=0.3)
axes[0].set_title(f'K-Means (k={optimal_k})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('PC1', fontsize=11)
axes[0].set_ylabel('PC2', fontsize=11)
axes[0].legend(fontsize=9)

# DBSCAN
unique_labels = sorted(df_cluster['cluster_dbscan'].unique())
db_colors = plt.cm.Set2(np.linspace(0, 1, max(len(unique_labels), 2)))
for idx, label in enumerate(unique_labels):
    mask = df_cluster['cluster_dbscan'] == label
    if label == -1:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                        c='gray', marker='x', s=20, alpha=0.4, label='Ruido')
    else:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                        c=[db_colors[idx]], label=f'Cluster {label}',
                        alpha=0.6, s=40, edgecolors='white', linewidth=0.3)
axes[1].set_title(f'DBSCAN ({n_clusters_db} clusters, {n_noise} ruido)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('PC1', fontsize=11)
axes[1].set_ylabel('PC2', fontsize=11)
axes[1].legend(fontsize=9)

fig.suptitle('Comparación K-Means vs DBSCAN (PCA 2D)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Perfilamiento de Clusters

Para cada cluster de K-Means, analizamos los valores medios de las features clave,
creamos **gráficos radar** y cruzamos con el tipo de vehículo para dar un **nombre de negocio**
a cada segmento.

In [ ]:
# Media por cluster (valores originales)
profile = df_cluster.groupby('cluster_kmeans')[cluster_features].mean()
print("Perfil medio por cluster (K-Means):")
display(profile.round(2))

In [ ]:
# Gráfico radar (spider chart) por cluster
def radar_chart(data, categories, title, ax, colors=None):
    """Crea un gráfico radar en el eje dado."""
    N = len(categories)
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]  # cerrar el polígono

    ax.set_theta_offset(pi / 2)
    ax.set_theta_direction(-1)
    ax.set_rlabel_position(0)

    plt.xticks(angles[:-1], categories, size=8)

    if colors is None:
        colors = plt.cm.Set2(np.linspace(0, 1, len(data)))

    for idx, (label, values) in enumerate(data.items()):
        vals = values.tolist()
        vals += vals[:1]
        ax.plot(angles, vals, 'o-', linewidth=2, label=label, color=colors[idx])
        ax.fill(angles, vals, alpha=0.1, color=colors[idx])

    ax.set_title(title, size=13, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

# Normalizar perfiles a [0, 1] para el radar
profile_norm = (profile - profile.min()) / (profile.max() - profile.min() + 1e-10)

# Limitar a las top features más discriminantes (max 8 para legibilidad)
feature_std = profile_norm.std()
top_features = feature_std.nlargest(min(8, len(cluster_features))).index.tolist()

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(polar=True))
radar_data = {f'Cluster {i}': profile_norm.loc[i, top_features] for i in range(optimal_k)}
radar_chart(radar_data, top_features, 'Perfil de Clusters K-Means', ax)
plt.tight_layout()
plt.show()

In [ ]:
# Crosstab cluster vs vehicle_type
if 'vehicle_type' in df.columns:
    # Alinear índices
    df_aligned = df.loc[df_cluster.index].copy()
    df_aligned['cluster_kmeans'] = df_cluster['cluster_kmeans'].values

    ct = pd.crosstab(df_aligned['cluster_kmeans'], df_aligned['vehicle_type'],
                     margins=True, margins_name='Total')
    print("Tabla cruzada: Cluster vs Tipo de Vehículo")
    display(ct)

    # Visualizar
    ct_no_total = ct.drop('Total', axis=0).drop('Total', axis=1)
    ct_pct = ct_no_total.div(ct_no_total.sum(axis=1), axis=0) * 100

    fig, ax = plt.subplots(figsize=(10, 5))
    ct_pct.plot(kind='bar', stacked=True, ax=ax,
                color=[vtype_colors.get(c, '#95a5a6') for c in ct_pct.columns],
                edgecolor='white')
    ax.set_title('Composición por tipo de vehículo en cada cluster', fontsize=13, fontweight='bold')
    ax.set_xlabel('Cluster', fontsize=11)
    ax.set_ylabel('Porcentaje (%)', fontsize=11)
    ax.legend(title='Tipo vehículo', bbox_to_anchor=(1.05, 1))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Columna 'vehicle_type' no disponible para crosstab.")

In [ ]:
# Asignar nombres de negocio a cada cluster basados en su perfil
print("=" * 60)
print("NOMBRES DE NEGOCIO SUGERIDOS PARA CADA SEGMENTO")
print("=" * 60)

# Generar descripciones basadas en las features más altas/bajas de cada cluster
for i in range(optimal_k):
    cluster_profile = profile_norm.loc[i]
    top_high = cluster_profile.nlargest(3).index.tolist()
    top_low = cluster_profile.nsmallest(3).index.tolist()

    # Nombres sugeridos según patrones comunes
    segment_names = [
        "Conductores conservadores urbanos",
        "Conductores eficientes de carretera",
        "Conductores agresivos frecuentes",
        "Conductores mixtos moderados",
        "Conductores nocturnos de alto consumo",
        "Conductores deportivos de alto rendimiento",
        "Conductores ecológicos conscientes",
        "Conductores ocasionales",
        "Viajeros de larga distancia"
    ]
    name = segment_names[i] if i < len(segment_names) else f"Segmento {i}"

    print(f"\nCluster {i}: \"{name}\"")
    print(f"  Features altas: {', '.join(top_high)}")
    print(f"  Features bajas: {', '.join(top_low)}")
    print(f"  Tamaño: {(df_cluster['cluster_kmeans'] == i).sum()} vehículos")

---
## 7. Análisis PCA

Examinamos cuántas componentes principales necesitamos para explicar la mayoría de la varianza
y qué features originales contribuyen más a cada componente (biplot).

In [ ]:
# PCA completo
n_components = min(len(cluster_features), X_scaled.shape[0])
pca_full = PCA(n_components=n_components, random_state=42)
pca_full.fit(X_scaled)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Varianza explicada individual
axes[0].bar(range(1, len(explained) + 1), explained * 100, color='#3498db', edgecolor='white')
axes[0].set_xlabel('Componente Principal', fontsize=12)
axes[0].set_ylabel('Varianza Explicada (%)', fontsize=12)
axes[0].set_title('Varianza por Componente', fontsize=13, fontweight='bold')

# Varianza acumulada
axes[1].plot(range(1, len(cumulative) + 1), cumulative * 100, 'ro-', linewidth=2, markersize=8)
axes[1].axhline(y=80, color='gray', linestyle='--', alpha=0.7, label='80% varianza')
axes[1].axhline(y=95, color='gray', linestyle=':', alpha=0.7, label='95% varianza')
axes[1].set_xlabel('Número de Componentes', fontsize=12)
axes[1].set_ylabel('Varianza Acumulada (%)', fontsize=12)
axes[1].set_title('Varianza Acumulada', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

fig.suptitle('Análisis de Componentes Principales', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Cuántos componentes para 80% y 95%
n_80 = np.argmax(cumulative >= 0.80) + 1
n_95 = np.argmax(cumulative >= 0.95) + 1
print(f"Componentes para 80% varianza: {n_80}")
print(f"Componentes para 95% varianza: {n_95}")

In [ ]:
# Biplot: visualizar feature loadings en PC1 vs PC2
loadings = pca_2d.components_.T  # (n_features, 2)
feature_names = cluster_features

fig, ax = plt.subplots(figsize=(12, 9))

# Scatter de observaciones (coloreadas por cluster)
for i in range(optimal_k):
    mask = df_cluster['cluster_kmeans'] == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=[cluster_colors[i]], alpha=0.3, s=20, label=f'Cluster {i}')

# Flechas de loadings
scale = max(abs(X_pca[:, :2]).max()) * 0.8
for j, feat in enumerate(feature_names):
    ax.arrow(0, 0, loadings[j, 0] * scale, loadings[j, 1] * scale,
             head_width=0.1, head_length=0.05, fc='#e74c3c', ec='#c0392b', linewidth=1.5)
    ax.text(loadings[j, 0] * scale * 1.12, loadings[j, 1] * scale * 1.12,
            feat, fontsize=9, ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', edgecolor='orange', alpha=0.8))

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
ax.set_title('Biplot PCA: Observaciones + Loadings de Features', fontsize=14, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
ax.legend(fontsize=9, loc='best')
plt.tight_layout()
plt.show()

In [ ]:
# Interpretación: qué features definen cada componente
loadings_df = pd.DataFrame(
    pca_full.components_[:min(4, n_components)].T,
    columns=[f'PC{i+1}' for i in range(min(4, n_components))],
    index=cluster_features
)

print("Loadings de las primeras componentes principales:")
display(loadings_df.round(3))

for i in range(min(3, n_components)):
    pc = loadings_df[f'PC{i+1}']
    top_pos = pc.nlargest(3)
    top_neg = pc.nsmallest(3)
    print(f"\nPC{i+1} ({explained[i]*100:.1f}% varianza):")
    print(f"  Contribución positiva: {', '.join([f'{k} ({v:.3f})' for k, v in top_pos.items()])}")
    print(f"  Contribución negativa: {', '.join([f'{k} ({v:.3f})' for k, v in top_neg.items()])}")

---
## 8. Resumen y Conclusiones

### Segmentos descubiertos

Mediante K-Means y DBSCAN identificamos segmentos naturales de clientes que combinan
patrones de conducción con características demográficas. Los principales hallazgos son:

- **K-Means** encontró segmentos bien definidos con separación medida por silueta.
- **DBSCAN** confirmó la estructura general e identificó puntos atípicos (outliers).
- **PCA** reveló las dimensiones principales que definen la variabilidad de los clientes.

### Recomendaciones de negocio

1. **Marketing personalizado**: Adaptar mensajes y campañas según el segmento del cliente.
2. **Diseño de producto**: Identificar qué características del vehículo valora cada segmento.
3. **Retención**: Monitorear satisfacción por segmento y actuar proactivamente.
4. **Pricing**: Ajustar ofertas de mantenimiento/seguros al perfil de conducción.

### Siguiente paso

> En el notebook [02 - Clustering de Patrones de Conducción](02_driving_pattern_clustering.ipynb)
> profundizamos exclusivamente en los patrones de conducción derivados de la telemetría,
> sin variables demográficas, para obtener perfiles de manejo más granulares.